# Listening Past the Noise — Inference

## Pipeline overview
1. **Artifact loading:** Locates and loads `meta_and_encoders.joblib`, `models.joblib`, and `train_refs.npz` exported by the training notebook.
2. **Test feature extraction:** Applies the same 6-view acoustic extraction function used during training to all recordings in `test.csv` / `audio/test/`, imputing any missing values with the saved training medians.
3. **Patch transformation:** Encodes test short-window patches into BoAW, VLAD, and Fisher vector representations using the pre-fitted scalers, k-means codebooks, PCA projections, and GMMs (transform only, no refitting).
4. **Kernel computation and prediction:** Computes test-against-train linear and RBF kernels using the saved training reference matrices, generates probability predictions from all 9 saved streams (including similarity-weighted verified 16-NN), and combines them with the fixed ensemble weights.
5. **Calibration and submission:** Divides out the saved OOF class marginal (`pi_src`), applies temperature scaling (`temp_opt`) and damped EM test-prior adjustment (`power = 0.50`), and writes the top-3 predicted categories per recording to `/kaggle/working/submission.csv`.

## Execution
Attach both the `project4` dataset and the saved output of the training notebook as inputs, then run all cells (`Save Version -> Save & Run All`).

In [ ]:
import os, gc, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import joblib
from scipy.signal import periodogram
from scipy.spatial.distance import cdist
from joblib import Parallel, delayed
from sklearn.metrics.pairwise import linear_kernel, rbf_kernel

warnings.filterwarnings("ignore")
T_GLOBAL = time.time()

# Locate saved training artifacts
SEARCH_ROOTS = [Path(os.environ.get("P4_ARTIFACTS", ""))] if os.environ.get("P4_ARTIFACTS") else []
SEARCH_ROOTS += [Path("/kaggle/input"), Path("work_step2b_train"), Path(".")]
meta_hits = []
for r in SEARCH_ROOTS:
    if r.exists():
        meta_hits.extend(list(r.rglob("meta_and_encoders.joblib")))
        if meta_hits:
            break
assert len(meta_hits) >= 1, "meta_and_encoders.joblib not found in input directories"
ART_DIR = meta_hits[0].parent

# Locate test data
if Path("/kaggle/input").exists():
    INPUT_ROOT = Path(os.environ.get("P4_INPUT", "/kaggle/input"))
    WORK = Path(os.environ.get("P4_WORK", "/kaggle/working"))
else:
    INPUT_ROOT = Path(os.environ.get("P4_INPUT", "project4"))
    WORK = Path(os.environ.get("P4_WORK", "work_step2b_infer"))
WORK.mkdir(parents=True, exist_ok=True)
N_JOBS = int(os.environ.get("P4_JOBS", -1))


test_csvs = [p for p in INPUT_ROOT.rglob("test.csv") if ART_DIR not in p.parents]
assert len(test_csvs) >= 1, f"Could not find test.csv under {INPUT_ROOT}"
DATA = test_csvs[0].parent
test_df = pd.read_csv(DATA / "test.csv", dtype={"id": str})
assert test_df["id"].is_unique

sample_csv = DATA / "sample_submission.csv"
sample_df = pd.read_csv(sample_csv, dtype={"id": str}) if sample_csv.exists() else test_df[["id"]].copy()

test_audio_hits = [p for p in DATA.rglob("audio/test") if p.is_dir()]
if not test_audio_hits:
    test_audio_hits = [p for p in INPUT_ROOT.rglob("audio/test") if p.is_dir() and ART_DIR not in p.parents]
assert len(test_audio_hits) >= 1, f"Could not find audio/test directory under {DATA}"
TEST_AUDIO = test_audio_hits[0]

print(f"artifacts: {ART_DIR} | test dir: {DATA} ({len(test_df)} clips)")

## Feature extraction and patch transform

In [ ]:
SR, N_FFT, HOP = 22050, 2048, 512
MEL32_2048 = librosa.filters.mel(sr=SR, n_fft=2048, n_mels=32).astype(np.float32)
MEL24_1024 = librosa.filters.mel(sr=SR, n_fft=1024, n_mels=24).astype(np.float32)
MEL40_1024 = librosa.filters.mel(sr=SR, n_fft=1024, n_mels=40).astype(np.float32)
MEL16_1024 = librosa.filters.mel(sr=SR, n_fft=1024, n_mels=16).astype(np.float32)

def count_onsets(env, sr, hop):
    if not np.isfinite(env).all() or not np.any(env):
        return 0
    normalized = env - env.min()
    normalized = normalized / (normalized.max() + np.finfo(env.dtype).tiny)
    pre_max = int(0.03 * sr // hop)
    pre_avg = int(0.10 * sr // hop)
    post_avg = pre_avg + 1
    wait = pre_max
    count = 0
    previous = -wait - 1
    for i, value in enumerate(normalized):
        if i <= previous + wait:
            continue
        peak_window = normalized[max(0, i - pre_max):i + 1]
        avg_window = normalized[max(0, i - pre_avg):i + post_avg]
        if value == peak_window.max() and value >= avg_window.mean() + 0.07:
            count += 1
            previous = i
    return count

def mask_slice(M, mask):
    n = min(M.shape[1], len(mask))
    m = mask[:n]
    if m.sum() < 2:
        m = np.ones(n, bool)
    return M[:, :n][:, m]

def make_window_patches(mat, widths=(8, 32)):
    views = []
    for w in widths:
        h = w // 2
        fr = np.pad(mat, ((0, max(0, w - len(mat))), (0, 0)), mode="edge")
        starts = list(range(0, len(fr) - w + 1, h))
        if starts[-1] != len(fr) - w:
            starts.append(len(fr) - w)
        bl = np.stack([fr[i:i + w] for i in starts])
        hf = w // 2
        views.append(np.hstack([bl.mean(1), bl.std(1), bl[:, hf:].mean(1) - bl[:, :hf].mean(1)]).astype(np.float32))
    return views

def extract_all_views_one(path):
    y, sr = sf.read(str(path), dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    if len(y) == 0:
        y = np.zeros(N_FFT, dtype=np.float32)
    dur = len(y) / sr
    y_pad = np.pad(y, (0, N_FFT - len(y))) if len(y) < N_FFT else y

    # Recording-level features
    S = np.abs(librosa.stft(y_pad, n_fft=N_FFT, hop_length=HOP))
    P2 = S ** 2
    rms  = librosa.feature.rms(y=y_pad, frame_length=N_FFT, hop_length=HOP)
    zcr  = librosa.feature.zero_crossing_rate(y=y_pad, frame_length=N_FFT, hop_length=HOP)
    mel128 = librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=128)
    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel128), n_mfcc=20)
    chroma = librosa.feature.chroma_stft(S=P2, sr=sr)
    cent = librosa.feature.spectral_centroid(S=S, sr=sr)
    bw   = librosa.feature.spectral_bandwidth(S=S, sr=sr)
    roll = librosa.feature.spectral_rolloff(S=S, sr=sr)
    con  = librosa.feature.spectral_contrast(S=S, sr=sr)
    ms = lambda M: np.concatenate([M.mean(axis=1), M.std(axis=1)])
    base = np.concatenate([ms(mfcc), ms(chroma), ms(cent), ms(bw), ms(roll), ms(zcr), ms(rms), [dur]])
    contrast = ms(con)
    r1 = rms[0]; T = S.shape[1]
    act = r1 >= max(np.percentile(r1, 30), 0.02 * r1.max())
    if act.sum() < 3:
        act = np.ones(T, bool)
    lm = librosa.power_to_db(librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=40))
    lma, mfa = lm[:, act], mfcc[:, act]
    dm = librosa.feature.delta(mfcc, width=9 if T >= 9 else 3, mode="nearest")
    flat = librosa.feature.spectral_flatness(S=S)[0]
    env = librosa.onset.onset_strength(S=lm, sr=sr, hop_length=HOP)
    n_on = count_onsets(env, sr, HOP)
    lr = np.log(r1 + 1e-8)
    rich = np.concatenate([
        lma.mean(1), lma.std(1), np.percentile(lma, 90, axis=1), lm.max(1),
        mfa.mean(1), mfa.std(1), dm.std(1),
        [flat.mean(), flat.std(), env.mean(), env.std()],
        np.percentile(lr, [10, 50, 90]),
        [act.mean(), np.log(dur + 1e-3), n_on / max(dur, 1e-3)]
    ])
    f_rich = np.concatenate([base, contrast, rich]).astype(np.float32)

    # Harmonic-percussive features
    raw_rms = float(np.sqrt(np.mean(y * y)) + 1e-8)
    raw_peak = float(np.max(np.abs(y)) + 1e-8)
    S_n = S / raw_peak
    P_n = S_n ** 2
    S_h, S_p = librosa.decompose.hpss(S_n, margin=2.0)
    P_h, P_p = S_h ** 2, S_p ** 2
    fe = P_n.sum(axis=0)
    fe_db = 10.0 * np.log10(fe + 1e-10)
    act20 = fe_db >= (fe_db.max() - 20.0)
    if act20.sum() < 4: act20 = fe >= np.percentile(fe, 70)
    if act20.sum() < 2: act20 = np.ones(len(fe), bool)
    top20 = fe >= np.percentile(fe, 80)
    if top20.sum() < 2: top20 = np.ones(len(fe), bool)
    h_e = P_h.sum(0); p_e = P_p.sum(0); t_e = fe + 1e-10
    h_rat = h_e / t_e; p_rat = p_e / t_e
    mel_h32 = librosa.power_to_db(MEL32_2048 @ P_h, ref=np.max)
    mel_p32 = librosa.power_to_db(MEL32_2048 @ P_p, ref=np.max)
    mfcc_h = librosa.feature.mfcc(S=mel_h32, n_mfcc=16)
    mfcc_p = librosa.feature.mfcc(S=mel_p32, n_mfcc=16)
    cent_h = librosa.feature.spectral_centroid(S=S_h, sr=sr)[0]
    cent_p = librosa.feature.spectral_centroid(S=S_p, sr=sr)[0]
    flat_h = librosa.feature.spectral_flatness(S=S_h + 1e-8)[0]
    flat_p = librosa.feature.spectral_flatness(S=S_p + 1e-8)[0]
    def stats3(M, mask):
        A = mask_slice(M, mask)
        return np.concatenate([A.mean(1), A.std(1), np.percentile(A, 90, axis=1)])
    f_hpss = np.concatenate([
        [h_rat.mean(), h_rat.std(), h_rat[act20].mean(), p_rat.mean(), p_rat.std(), p_rat[act20].mean(),
         (h_e.sum() / (t_e.sum() + 1e-10)), (p_e.sum() / (t_e.sum() + 1e-10)),
         cent_h[act20].mean(), cent_h[act20].std(), cent_p[act20].mean(), cent_p[act20].std(),
         flat_h[act20].mean(), flat_p[act20].mean(),
         np.log(raw_rms), np.log(raw_peak), raw_peak / raw_rms, act20.mean()],
        stats3(mfcc_h, act20), stats3(mfcc_p, act20),
        mask_slice(mel_h32, top20).mean(1), mask_slice(mel_p32, top20).mean(1),
    ]).astype(np.float32)

    # Modulation spectrum features
    width_m = 1024
    hop_m = max(1, round(sr * 0.010))
    yp_m = np.pad(y, (width_m // 2, width_m // 2))
    frames_m = np.lib.stride_tricks.sliding_window_view(yp_m, width_m)[::hop_m]
    spec_m = np.abs(np.fft.rfft(frames_m * np.hanning(width_m), axis=1)) ** 2
    hz = np.fft.rfftfreq(width_m, 1.0 / sr)
    mel_edges = np.linspace(0, np.log1p(min(11000, sr / 2) / 700), 26)
    edges = 700 * np.expm1(mel_edges)
    bank = np.maximum(0, np.minimum(
        (hz[:, None] - edges[:-2]) / np.diff(edges)[:-1],
        (edges[2:] - hz[:, None]) / np.diff(edges)[1:]))
    energy = spec_m @ bank
    envelope = np.log1p(100 * energy / max(energy.mean(), 1e-10))
    centered = envelope - envelope.mean(axis=0)
    scales = np.sqrt(np.mean(centered ** 2, axis=0))
    norm = centered / np.maximum(scales, 1e-6)
    corr = norm.T @ norm / len(norm)
    freq, power = periodogram(centered, fs=sr / hop_m, axis=0, nfft=max(256, len(centered)), detrend=False, scaling="density")
    bands = [(0.5, 2), (2, 4), (4, 8), (8, 16), (16, 32), (32, 50)]
    modulation = [np.log1p(power[(freq >= lo) & (freq < hi)].sum(0) * freq[1]) for lo, hi in bands]
    f_mod = np.concatenate([envelope.mean(0), scales, corr[np.triu_indices(24, k=1)], *modulation]).astype(np.float32)

    # Short-window spectrogram patches
    hop_p = max(1, round(sr * 0.012))
    S1 = np.abs(librosa.stft(y, n_fft=1024, hop_length=hop_p))
    P1 = S1 ** 2
    lm40 = librosa.power_to_db(MEL40_1024 @ P1, ref=np.max).T
    p_std = make_window_patches(lm40, (8, 32))

    S1_h, S1_p = librosa.decompose.hpss(S1, margin=2.0)
    lm_h = librosa.power_to_db(MEL24_1024 @ (S1_h ** 2), ref=np.max).T
    lm_p = librosa.power_to_db(MEL24_1024 @ (S1_p ** 2), ref=np.max).T
    p_hpss = make_window_patches(np.hstack([lm_h, lm_p]), (8, 32))

    # Harmonic-percussive modulation features
    eh = (MEL16_1024 @ (S1_h ** 2)).T
    ep = (MEL16_1024 @ (S1_p ** 2)).T
    env_h = np.log1p(100.0 * eh / max(eh.mean(), 1e-10))
    env_p = np.log1p(100.0 * ep / max(ep.mean(), 1e-10))
    ch = env_h - env_h.mean(0); cp = env_p - env_p.mean(0)
    sh = np.sqrt(np.mean(ch ** 2, axis=0)); sp = np.sqrt(np.mean(cp ** 2, axis=0))
    nh = ch / np.maximum(sh, 1e-6); np_ = cp / np.maximum(sp, 1e-6)
    corr_h = (nh.T @ nh) / len(nh); corr_p = (np_.T @ np_) / len(np_)
    corr_hp_diag = np.mean(nh * np_, axis=0)
    fs_env = sr / float(hop_p)
    fq, pw_h = periodogram(ch, fs=fs_env, axis=0, nfft=max(256, len(ch)), detrend=False, scaling="density")
    _,  pw_p = periodogram(cp, fs=fs_env, axis=0, nfft=max(256, len(cp)), detrend=False, scaling="density")
    mbands_hp = [(0.5, 2), (2, 4), (4, 8), (8, 16), (16, 32), (32, 41.5)]
    mod_h = [np.log1p(pw_h[(fq >= lo) & (fq < hi)].sum(0) * fq[1]) for lo, hi in mbands_hp]
    mod_p = [np.log1p(pw_p[(fq >= lo) & (fq < hi)].sum(0) * fq[1]) for lo, hi in mbands_hp]
    f_mod_hpss = np.concatenate([
        env_h.mean(0), sh, env_p.mean(0), sp,
        corr_h[np.triu_indices(16, k=1)], corr_p[np.triu_indices(16, k=1)], corr_hp_diag,
        *mod_h, *mod_p
    ]).astype(np.float32)

    return f_rich, f_hpss, f_mod, f_mod_hpss, p_std, p_hpss

def transform_patch_family(patch_list, encoders):

    boaw_v, vlad_v = [], []
    n_gmm_seeds = len(encoders[0]["gmms"])
    fv_by_seed = [[] for _ in range(n_gmm_seeds)]
    for view, enc in enumerate(encoders):
        scaler = enc["scaler"]
        c96 = enc["c96"]
        c32 = enc["c32"]
        def _boaw(rec):
            fr = scaler.transform(rec[view]).astype(np.float32)
            d = cdist(fr, c96, "sqeuclidean")
            nr = np.argpartition(d, 2, axis=1)[:, :3]
            w = 1.0 / np.maximum(np.take_along_axis(d, nr, axis=1), 1e-8)
            w /= w.sum(axis=1, keepdims=True)
            h = np.bincount(nr.ravel(), weights=w.ravel(), minlength=len(c96))
            return np.sqrt(h / max(h.sum(), 1e-12)).astype(np.float32)
        boaw_v.append(np.vstack([_boaw(r) for r in patch_list]))

        def _vlad(rec):
            fr = scaler.transform(rec[view]).astype(np.float32)
            asgn = cdist(fr, c32, "sqeuclidean").argmin(axis=1)
            sums = np.zeros_like(c32)
            np.add.at(sums, asgn, fr - c32[asgn])
            sums /= np.maximum(np.linalg.norm(sums, axis=1, keepdims=True), 1e-8)
            v = np.sign(sums.ravel()) * np.sqrt(np.abs(sums.ravel()))
            return (v / max(np.linalg.norm(v), 1e-8)).astype(np.float32)
        vlad_v.append(np.stack([_vlad(r) for r in patch_list]))

        for si, g_enc in enumerate(enc["gmms"]):
            pca = g_enc["pca"]; gmm = g_enc["gmm"]
            mu, var, wt = gmm.means_, gmm.covariances_, gmm.weights_
            def _fv(rec):
                x = pca.transform(scaler.transform(rec[view]))
                q = gmm.predict_proba(x)
                mass = q.sum(axis=0)[:, None]
                mom = q.T @ x
                norm = len(x) * np.sqrt(wt[:, None])
                f1 = (mom - mass * mu) / (norm * np.sqrt(var))
                f2 = ((q.T @ (x * x) - 2 * mu * mom + mass * mu**2) / var - mass) / (np.sqrt(2) * norm)
                v = np.concatenate([f1.ravel(), f2.ravel()])
                v = np.sign(v) * np.sqrt(np.abs(v))
                return (v / max(np.linalg.norm(v), 1e-8)).astype(np.float32)
            fv_by_seed[si].append(np.stack([_fv(r) for r in patch_list]))
    return (
        np.hstack(boaw_v).astype(np.float32),
        np.hstack(vlad_v).astype(np.float32),
        [np.hstack(fv_by_seed[si]).astype(np.float32) for si in range(n_gmm_seeds)],
    )

## Prediction and submission

In [ ]:
t_load = time.time()
meta   = joblib.load(ART_DIR / "meta_and_encoders.joblib")
models = joblib.load(ART_DIR / "models.joblib")
refs   = np.load(ART_DIR / "train_refs.npz")
assert meta["feature_version"] == "step2b-v1", f"Feature version mismatch: {meta['feature_version']}"

classes = meta["classes"]
cls_idx = meta["cls_idx"]
K       = meta["K"]
med_r   = meta["med_r"]
n_vtr   = meta["n_vtr"]
yv      = meta["yv"]
W_FIXED = meta["W_FIXED"]
pi_src  = meta["pi_src"]
temp_opt = meta["temp_opt"]
print(f"Loaded artifacts in {time.time() - t_load:.1f}s (K={K}, temp={temp_opt:.2f})", flush=True)

# Extract test features
t_ext = time.time()
test_paths = [TEST_AUDIO / f"{i}.wav" for i in test_df["id"]]
print(f"Extracting features for {len(test_paths)} test clips...", flush=True)
res = Parallel(n_jobs=N_JOBS, batch_size=8, verbose=5)(delayed(extract_all_views_one)(p) for p in test_paths)
print(f"Feature extraction: {time.time() - t_ext:.1f}s", flush=True)

X_rich_te = np.vstack([r[0] for r in res]).astype(np.float32)
X_hpss_te = np.vstack([r[1] for r in res]).astype(np.float32)
X_mod_te  = np.vstack([r[2] for r in res]).astype(np.float32)
X_mh_te   = np.vstack([r[3] for r in res]).astype(np.float32)
std_patches_te  = [r[4] for r in res]
hpss_patches_te = [r[5] for r in res]
del res; gc.collect()

# Impute missing values using training medians
X_rich_te = np.where(np.isfinite(X_rich_te), X_rich_te, np.nan)
X_rich_te = np.where(np.isnan(X_rich_te), med_r, X_rich_te).astype(np.float32)
X_hpss_te = np.nan_to_num(X_hpss_te).astype(np.float32)
X_mod_te  = np.nan_to_num(X_mod_te).astype(np.float32)
X_mh_te   = np.nan_to_num(X_mh_te).astype(np.float32)
X_tab_te  = np.hstack([X_rich_te, X_hpss_te]).astype(np.float32)

# Transform test patches
t_enc = time.time()
boaw_s_te, vlad_s_te, [fv_s_te] = transform_patch_family(std_patches_te, meta["std_encoders"])
del std_patches_te; gc.collect()
boaw_h_te, vlad_h_te, [fv_h1_te, fv_h2_te] = transform_patch_family(hpss_patches_te, meta["hpss_encoders"])
del hpss_patches_te; gc.collect()
boaw_both_te = np.hstack([boaw_s_te, boaw_h_te]).astype(np.float32)
print(f"Patch encoding: {time.time() - t_enc:.1f}s", flush=True)

# Compute test-vs-train kernels and predict
t_pred = time.time()
K_fv_h_te = (0.5 * linear_kernel(fv_h1_te, refs["fv_h1_all"]) + 0.5 * linear_kernel(fv_h2_te, refs["fv_h2_all"])).astype(np.float32)
K_fv_s_te = linear_kernel(fv_s_te, refs["fv_s_all"]).astype(np.float32)
K_vl_h_te = linear_kernel(vlad_h_te, refs["vl_h_all"]).astype(np.float32)
K_vl_s_te = linear_kernel(vlad_s_te, refs["vl_s_all"]).astype(np.float32)
K_bw_all_te = rbf_kernel(boaw_both_te, refs["bw_both_all"], gamma=meta["boaw_both_gamma"]).astype(np.float32)
K_bw_s_te   = rbf_kernel(boaw_s_te, refs["bw_s_all"], gamma=meta["boaw_s_gamma"]).astype(np.float32)

Xr_te_s   = models["sc_r"].transform(X_rich_te).astype(np.float32)
Xh_te_s   = models["sc_h"].transform(X_hpss_te).astype(np.float32)
Xm_te_s   = models["sc_m"].transform(X_mod_te).astype(np.float32)
Xmha_te_s = models["sc_mha"].transform(X_mh_te).astype(np.float32)

Kr_te   = rbf_kernel(Xr_te_s, refs["Xr_all_s"], gamma=1.0 / Xr_te_s.shape[1]).astype(np.float32)
Kh_te   = rbf_kernel(Xh_te_s, refs["Xh_all_s"], gamma=1.0 / Xh_te_s.shape[1]).astype(np.float32)
Km_te   = rbf_kernel(Xm_te_s, refs["Xm_all_s"], gamma=1.0 / Xm_te_s.shape[1]).astype(np.float32)
Kmha_te = rbf_kernel(Xmha_te_s, refs["Xmha_all_s"], gamma=1.0 / Xmha_te_s.shape[1]).astype(np.float32)

# Multi-kernel SVM
K_mkl_te = (0.24 * K_fv_h_te + 0.16 * K_fv_s_te + 0.14 * K_vl_h_te + 0.10 * K_vl_s_te +
            0.06 * K_bw_all_te + 0.11 * Kr_te + 0.10 * Kh_te + 0.09 * Km_te)
p_mkl = models["clf_mkl"].predict_proba(K_mkl_te)

# Multinomial logistic regression
X_jlr_te = np.hstack([
    np.sqrt(0.28) * fv_h1_te, np.sqrt(0.28) * fv_h2_te, np.sqrt(0.26) * fv_s_te,
    np.sqrt(0.18) * vlad_h_te, np.sqrt(0.14) * vlad_s_te,
]).astype(np.float32)
p_jlr = models["clf_jlr"].predict_proba(X_jlr_te)

# HPSS kernel SVM
p_hpss_k = models["clf_hpss_k"].predict_proba(0.60 * K_fv_h_te + 0.40 * K_vl_h_te)

# Log-mel kernel SVM
p_std_k = models["clf_std_k"].predict_proba(0.55 * K_fv_s_te + 0.30 * K_vl_s_te + 0.15 * K_bw_s_te)

# Tabular ensemble
p_tab = np.zeros((len(test_df), K), dtype=np.float64)
for m, a in models["tab_models"]:
    p_tab[:, m.classes_] += a * m.predict_proba(X_tab_te)

# Modulation SVM
p_mod = 0.50 * models["clf_mod_k"].predict_proba(models["sc_mk"].transform(X_mod_te)) + 0.50 * models["clf_mod_a"].predict_proba(Km_te)

# HPSS modulation SVM
p_mod_hpss = 0.50 * models["clf_mhk"].predict_proba(models["sc_mhk"].transform(X_mh_te)) + 0.50 * models["clf_mha"].predict_proba(Kmha_te)

# Shrunk LDA
p_lda = np.zeros((len(test_df), K), dtype=np.float64)
p_lda[:, models["clf_lda"].classes_.astype(int)] = models["clf_lda"].predict_proba(X_tab_te)

# Similarity-weighted verified 16-NN
S_v = K_mkl_te[:, :n_vtr]
cnt_tr = np.bincount(yv, minlength=K)
inv_c = (1.0 / np.maximum(cnt_tr, 1))[yv]
p_knn = np.zeros((len(test_df), K), dtype=np.float64)
for i in range(len(test_df)):
    sims = S_v[i]
    top = np.argpartition(-sims, 16)[:16]
    w_n = np.exp(15.0 * (sims[top] - sims[top].max())) * inv_c[top]
    votes = np.bincount(yv[top], weights=w_n, minlength=K)
    p_knn[i] = votes / np.maximum(votes.sum(), 1e-12)

streams_te = dict(mkl=p_mkl, jlr=p_jlr, hpss_k=p_hpss_k, std_k=p_std_k, tab=p_tab, mod=p_mod, mod_hpss=p_mod_hpss, lda=p_lda, knn=p_knn)
P_test_raw = sum(W_FIXED[nm] * streams_te[nm] for nm in W_FIXED)
print(f"Model inference: {time.time() - t_pred:.1f}s", flush=True)

# Calibration and prior adjustment
def sharpen(P, t):
    Q = np.clip(P, 1e-9, None) ** t
    return Q / Q.sum(1, keepdims=True)

def em_prior(P, iters=200, floor=0.25):
    pi = np.full(K, 1.0 / K)
    for _ in range(iters):
        Q = P * (pi * K)[None, :]
        Q /= Q.sum(1, keepdims=True)
        pi = Q.mean(0)
    pi = np.clip(pi, floor / K, None)
    return pi / pi.sum()

P_test_deb = P_test_raw / np.maximum(pi_src[None, :], 1e-6)
P_test_deb /= P_test_deb.sum(1, keepdims=True)
P_test_cal = sharpen(P_test_deb, temp_opt)

if len(test_df) >= 50:
    pi_test_hat = em_prior(P_test_cal)
    P_test_final = P_test_cal * (pi_test_hat ** 0.50)[None, :]
    P_test_final /= P_test_final.sum(1, keepdims=True)
    print(f"Estimated test prior range: {100*pi_test_hat.min():.2f}% - {100*pi_test_hat.max():.2f}%")
else:
    P_test_final = P_test_cal

# Save submission
top3 = np.argsort(-P_test_final, axis=1)[:, :3]
sub = pd.DataFrame({"id": test_df["id"], "label": [" ".join(classes[r]) for r in top3]})
if set(sample_df["id"]) == set(test_df["id"]):
    sub = sample_df[["id"]].merge(sub, on="id", how="left", validate="one_to_one")
assert sub.shape == (len(test_df), 2), f"Bad shape: {sub.shape}"
assert sub["label"].notna().all() and sub["id"].is_unique
assert all(len(set(s.split())) == 3 and all(c in cls_idx for c in s.split()) for s in sub["label"])

sub_path = WORK / "submission.csv"
sub.to_csv(sub_path, index=False, lineterminator="\n")
print(f"Saved {sub_path} {sub.shape} | elapsed: {(time.time() - T_GLOBAL)/60:.2f} min")
print(f"Unique top-1 classes: {sub['label'].str.split().str[0].nunique()}/{K}")
print(sub.head(10).to_string(index=False))

## Summary and evaluation

### Implementation flow (Submission 14 — Inference-Only)
1. **Artifact loading:** Auto-locates and loads `meta_and_encoders.joblib`, `models.joblib`, and `train_refs.npz` (`797.6 MB`, loads in `~1.8 s`).
2. **Test-only extraction & transformation:** Runs the identical 6-view extractor on `test.csv` / `audio/test/` only, imputes with saved training medians, and transforms patches using pre-fitted BoAW/VLAD/PCA+GMM encoders without refitting.
3. **Inference & calibration:** Computes test-vs-train kernels against saved training references, runs all 9 saved streams + verified 16-NN (`~15 s` for 1,592 clips), applies saved `pi_src` debiasing, `temp_opt`, and damped EM prior adjustment (`power = 0.50`), and outputs `/kaggle/working/submission.csv`.

### Changes from Submission 13 (Step 1) and all-in-one Submission 14
- Runs strictly inference-only with zero test-time training, reducing Kaggle 4-core CPU runtime from ~84 minutes to ~10 minutes.
- Uses dynamic test/artifact path discovery and drops hardcoded `1592` test-length checks for hidden private-test evaluation.

### Verification results
| Check | Result |
| :--- | :---: |
| Top-1 parity vs. all-in-one Submission 14 (`0.883` LB) | `99.81%` (`1,589 / 1,592`) |
| Output schema & class coverage | `1,592 × 2`, `41 / 41` unique top-1 classes |
| Estimated test prior range (Debiased EM) | `1.08% – 8.11%` |
| Inference runtime (Kaggle 4-core CPU) | `~10 min` (vs. `~84 min` full retrain) |